# * Mobile Devices

In [1]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

In [5]:
''' Input parameter '''

# op_dir = 'C:/Ruz/MyProject/Code/Jupyter/data/Warroom/Mobile Device/output'
op_dir = '../../data/Warroom/Mobile Device/output'
op_file = f'mobile_device_{str_curr_dt}'

print(f'\nParameter input...\n')
print(f'   -> op_dir: {op_dir}')
print(f'   -> op_file: {op_file}')


Parameter input...

   -> op_dir: ../../data/Warroom/Mobile Device/output
   -> op_file: mobile_device_20260312


### Step 1 : Import Data Source

In [4]:
''' Rawdata '''

# C:\Ruz\TrueCorp\VC 2026\Warroom\Mobile Devices\RAW_MSISDN_DEVICE_NANOCLUSTER-MKS_ARPU.csv

src_file = 'C:/Ruz/TrueCorp/VC 2026/Warroom/Mobile Devices/RAW_MSISDN_DEVICE_NANOCLUSTER-MKS_ARPU.csv'
# src_file = 'data/RAW_MSISDN_DEVICE_NANOCLUSTER-MKS_ARPU.csv'
src_df = pd.read_csv(src_file)

print(f'\nsrc_df : {src_df.shape[0]} rows, {src_df.shape[1]} columns')
src_df.tail(3)


src_df : 8045317 rows, 32 columns


,CUSTOMER,MSISDN,ACCOUNT_TYPE,ACTIVE_IND,AOU_DEVICE,ARPU_RANGE,CCAATT,SITENAME,REGION,PROVINCE_EN,...,FLAG_DEVICE_IOS-BELOW-IPHONE11,FLAG_4Yearold-luanch_Device,qk16_cen_latitude,qk16_cen_longitude,NN_ID,CLSTR_GRP,TRUEDTAC_MKS,HH,POP,ARPU
8045314,D,66633299855,PREPAID,Y,>18Months,0-99,700701.0,RCB1001,Central-West,Ratchaburi,...,N,N,13.696693,99.846497,700713-001,1,39.96,2105,5979,34.9985
8045315,D,66804381244,PREPAID,Y,>18Months,100-199,570906.0,CRI0517,North,Chiang Rai,...,N,N,20.434734,99.879456,570906-001,6,62.01,15735,34560,185.4206
8045316,D,66629693836,PREPAID,Y,>18Months,300-399,900311.0,SKA0115,South,Songkhla,...,N,Y,6.880073,100.719910,900311-001,4,42.24,1549,5836,329.3637


### Step 2 : Aggregate Data

In [31]:
# agg_df = src_df.query(" NN_ID == '210109-001' ").reset_index(drop=True)
agg_df = src_df.copy()

agg_df['#DEVICE_PER_NN'] = agg_df.groupby('NN_ID')['MSISDN'].transform('nunique')
agg_df['MD_DATA_PER_NN'] = agg_df.groupby('NN_ID')['DATA_GB_PER_MONTH'].transform('median')
agg_df['MD_ARPU_PER_NN'] = agg_df.groupby('NN_ID')['ARPU'].transform('median')

agg_df.head()

,CUSTOMER,MSISDN,ACCOUNT_TYPE,ACTIVE_IND,AOU_DEVICE,ARPU_RANGE,CCAATT,SITENAME,REGION,PROVINCE_EN,...,qk16_cen_longitude,NN_ID,CLSTR_GRP,TRUEDTAC_MKS,HH,POP,ARPU,#DEVICE_PER_NN,MD_DATA_PER_NN,MD_ARPU_PER_NN
0,T,66818898617,POSTPAID,Y,>18Months,700-799,100101.0,BKD1016,BMA-East,Bangkok,...,100.489197,100101-001,1,57.79,3952,9416,799.0,2482,7.230,224.19200
1,T,66935691556,POSTPAID,Y,>18Months,0-99,100102.0,BKDC005,BMA-East,Bangkok,...,100.500183,100103-001,1,57.01,1707,2991,0.0,1118,5.765,243.55695
2,T,66844342353,POSTPAID,Y,>18Months,300-399,100104.0,BKD0046,BMA-East,Bangkok,...,100.505676,100804-001,1,63.01,2368,3615,300.0,1227,2.830,212.28800
3,T,66834215218,POSTPAID,Y,>18Months,200-299,100107.0,BKD0015,BMA-East,Bangkok,...,100.505676,100110-001,1,55.14,1553,2862,250.0,1069,10.500,218.30625
4,T,66639453031,POSTPAID,Y,>18Months,300-399,100107.0,BKD7198,BMA-East,Bangkok,...,100.500183,100107-001,1,58.90,4597,8881,320.0,2635,5.390,200.00000


In [32]:
sample_nano_df = agg_df.query(" NN_ID == '210109-001' ").reset_index(drop=True)
sample_nano_df

,CUSTOMER,MSISDN,ACCOUNT_TYPE,ACTIVE_IND,AOU_DEVICE,ARPU_RANGE,CCAATT,SITENAME,REGION,PROVINCE_EN,...,qk16_cen_longitude,NN_ID,CLSTR_GRP,TRUEDTAC_MKS,HH,POP,ARPU,#DEVICE_PER_NN,MD_DATA_PER_NN,MD_ARPU_PER_NN
0,T,66958191295,POSTPAID,Y,>18Months,0-99,210109.0,RYG0043,East,Rayong,...,101.214294,210109-001,1,52.71,45640,46965,0.0000,11049,14.04,277.5
1,T,66638265624,POSTPAID,Y,>18Months,300-399,210109.0,RYG0045,East,Rayong,...,101.258240,210109-001,1,52.71,45640,46965,300.0000,11049,14.04,277.5
2,T,66825798524,POSTPAID,Y,>18Months,100-199,210109.0,RYG0085,East,Rayong,...,101.252747,210109-001,1,52.71,45640,46965,100.0000,11049,14.04,277.5
3,T,66863930698,POSTPAID,Y,>18Months,100-199,210109.0,RYG0201,East,Rayong,...,101.241760,210109-001,1,52.71,45640,46965,190.0000,11049,14.04,277.5
4,T,66954954319,POSTPAID,Y,>18Months,0-99,210109.0,RYG0203,East,Rayong,...,101.175842,210109-001,1,52.71,45640,46965,0.0000,11049,14.04,277.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11060,D,66993769778,PREPAID,Y,>18Months,600-699,210109.0,RYG0005,East,Rayong,...,101.186829,210109-001,1,52.71,45640,46965,626.0946,11049,14.04,277.5
11061,D,66627577439,PREPAID,Y,>18Months,200-299,210109.0,RYG0201,East,Rayong,...,101.241760,210109-001,1,52.71,45640,46965,231.7757,11049,14.04,277.5
11062,D,66635342328,PREPAID,Y,>18Months,0-99,210109.0,RYG6014,East,Rayong,...,101.241760,210109-001,1,52.71,45640,46965,51.9178,11049,14.04,277.5
11063,D,66805206526,PREPAID,Y,>18Months,200-299,210109.0,RYG0069,East,Rayong,...,101.219788,210109-001,1,52.71,45640,46965,289.7288,11049,14.04,277.5


In [33]:
# Group by 'store_id' and count unique 'product_id'
x = sample_nano_df.groupby('NN_ID')['MSISDN'].nunique()
x

NN_ID
210109-001    11049
Name: MSISDN, dtype: int64

In [34]:
src_df['FLAG_DVC-CONTRACT_EXP3MONTHS'].drop_duplicates()

0    1
Name: FLAG_DVC-CONTRACT_EXP3MONTHS, dtype: int64

In [35]:
src_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8045317 entries, 0 to 8045316
Data columns (total 32 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   CUSTOMER                        object 
 1   MSISDN                          int64  
 2   ACCOUNT_TYPE                    object 
 3   ACTIVE_IND                      object 
 4   AOU_DEVICE                      object 
 5   ARPU_RANGE                      object 
 6   CCAATT                          float64
 7   SITENAME                        object 
 8   REGION                          object 
 9   PROVINCE_EN                     object 
 10  DISTRICT_EN                     object 
 11  SUB_DISTRICT_EN                 object 
 12  IMEI_TAC                        int64  
 13  mkt_brand_name                  object 
 14  handset_model                   object 
 15  mkt_model_name                  object 
 16  DVC_OS                          object 
 17  DVC_TYPE                   

### Step 3 : Output file

In [36]:
# ''' Generate CSV file '''

# agg_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
# print(f'\n   -> Generate "{op_file}.csv" successfully')


   -> Generate "mobile_device_20260302.csv" successfully
